# Taller ANN Multiclass: EDA profundo + PCA + ANN

Este es mi cuaderno de trabajo para el taller. Documento la carga, auditoria, limpieza, PCA y una ANN multiclase para clasificar `Credit_Score` en las clases 0, 1 y 2.


## 1. Importación de dependencias

In [1]:
# Dependencias base para EDA, PCA y la ANN multiclase.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

np.random.seed(42)
tf.keras.utils.set_random_seed(42)
plt.style.use('ggplot')


## 2. Carga de datos


In [ ]:
# Cargamos el CSV directamente con pandas y definimos solo constantes minimas.
def find_workspace_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path(r'c:/Users/cremo/OneDrive/Documentos/GitHub/clase-ciencia-de-datos'),
    ]

    for candidate in candidates:
        if candidate is None:
            continue
        candidate = candidate.resolve()
        expected_file = candidate / 'notebooks' / 'Taller ANN Multiclass' / 'riesgo.csv'
        if expected_file.exists():
            return candidate

    raise FileNotFoundError('No fue posible localizar el archivo riesgo.csv en el workspace.')


WORKSPACE_ROOT = find_workspace_root()
WORKSHOP_DIR = WORKSPACE_ROOT / 'notebooks' / 'Taller ANN Multiclass'
DATA_PATH = WORKSHOP_DIR / 'riesgo.csv'
ARTIFACTS_DIR = WORKSHOP_DIR / 'artefactos_eda_pca'

raw_df = pd.read_csv(DATA_PATH, sep=';')
df = raw_df.copy()

target_col = 'Credit_Score'
id_cols = ['Customer_ID', 'Name', 'SSN']
loan_col = 'Type_of_Loan'

print(f'Archivo cargado: {DATA_PATH}')
print(f'Dimensiones iniciales: {df.shape[0]:,} filas x {df.shape[1]:,} columnas')
print('Tipos detectados por pandas:')
display(df.dtypes.rename('dtype').to_frame())

print('Distribución inicial del target:')
display(
    df[target_col]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis(target_col)
    .reset_index(name='frecuencia')
)

print('Vista previa del dataset original:')
display(df.head())


## 3. Auditoría y limpieza vectorizada


In [ ]:
# Auditoria inicial y limpieza usando solo pandas/numpy.
text_columns = df.select_dtypes(include=['object', 'string']).columns.tolist()

numeric_columns = [
    'Age',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Num_Bank_Accounts',
    'Num_Credit_Card',
    'Interest_Rate',
    'Num_of_Loan',
    'Delay_from_due_date',
    'Num_of_Delayed_Payment',
    'Changed_Credit_Limit',
    'Num_Credit_Inquiries',
    'Outstanding_Debt',
    'Credit_Utilization_Ratio',
    'Credit_History_Age',
    'Total_EMI_per_month',
    'Amount_invested_monthly',
    'Monthly_Balance',
]
count_columns = [
    'Num_Bank_Accounts',
    'Num_Credit_Card',
    'Num_of_Loan',
    'Num_of_Delayed_Payment',
    'Changed_Credit_Limit',
    'Num_Credit_Inquiries',
]
categorical_columns = ['Occupation', loan_col, 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']
base_categorical_columns = [column for column in categorical_columns if column != loan_col]

text_view_before = raw_df.astype('string')
quality_before = pd.DataFrame(
    {
        'column': raw_df.columns,
        'dtype': raw_df.dtypes.astype(str).to_numpy(),
        'missing_count': raw_df.isna().sum().to_numpy(),
        'missing_pct': (raw_df.isna().mean() * 100).to_numpy(),
        'blank_like_count': text_view_before.apply(lambda s: s.str.strip().eq('').fillna(False).sum()).to_numpy(),
        'NM_count': text_view_before.apply(lambda s: s.eq('NM').fillna(False).sum()).to_numpy(),
        'Not_Specified_count': text_view_before.apply(lambda s: s.eq('Not Specified').fillna(False).sum()).to_numpy(),
        'unique_values': raw_df.nunique(dropna=True).to_numpy(),
    }
).sort_values(['missing_count', 'blank_like_count'], ascending=False).reset_index(drop=True)

numeric_view_before = raw_df[numeric_columns + [target_col]].apply(pd.to_numeric, errors='coerce')
suspicious_before = pd.DataFrame(
    {
        'check': [
            'Age < 18',
            'Delay_from_due_date < 0',
            'Num_Bank_Accounts con decimales',
            'Num_Credit_Card con decimales',
            'Num_of_Loan con decimales',
            'Num_of_Delayed_Payment con decimales',
            'Changed_Credit_Limit con decimales',
            'Num_Credit_Inquiries con decimales',
        ],
        'count': [
            int(numeric_view_before['Age'].lt(18).fillna(False).sum()),
            int(numeric_view_before['Delay_from_due_date'].lt(0).fillna(False).sum()),
            int(((numeric_view_before['Num_Bank_Accounts'].dropna() % 1) != 0).sum()),
            int(((numeric_view_before['Num_Credit_Card'].dropna() % 1) != 0).sum()),
            int(((numeric_view_before['Num_of_Loan'].dropna() % 1) != 0).sum()),
            int(((numeric_view_before['Num_of_Delayed_Payment'].dropna() % 1) != 0).sum()),
            int(((numeric_view_before['Changed_Credit_Limit'].dropna() % 1) != 0).sum()),
            int(((numeric_view_before['Num_Credit_Inquiries'].dropna() % 1) != 0).sum()),
        ],
    }
).sort_values('count', ascending=False).reset_index(drop=True)

# Normalizamos texto y reemplazamos tokens especiales por NaN.
df[text_columns] = df[text_columns].astype('string').apply(lambda s: s.str.strip())
df[text_columns] = df[text_columns].replace({'': pd.NA, 'NM': pd.NA, 'Not Specified': pd.NA, 'nan': pd.NA, 'None': pd.NA})

# Convertimos variables numericas con coercion segura.
for column in numeric_columns + [target_col]:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Aproximamos conteos a enteros y dejamos Delay_from_due_date negativo como pagos adelantados.
df[count_columns] = df[count_columns].round().astype('Int64')

numeric_fill_values = df[numeric_columns].median(numeric_only=True)
df[numeric_columns] = df[numeric_columns].fillna(numeric_fill_values)
df[base_categorical_columns] = df[base_categorical_columns].fillna('Desconocido')

loan_series = df[loan_col].astype('string')
loan_series = loan_series.str.replace(', and ', ', ', regex=False)
loan_series = loan_series.str.replace(' and ', ', ', regex=False)
loan_series = loan_series.str.replace(';', ',', regex=False)
loan_series = loan_series.str.replace(r'\s+', ' ', regex=True)
loan_series = loan_series.str.replace(r'(^|,\s*)(NM|Not Specified)(?=,|$)', '', regex=True)
loan_series = loan_series.str.replace(r'\s*,\s*', ', ', regex=True)
loan_series = loan_series.str.replace(r'(,\s*){2,}', ', ', regex=True)
loan_series = loan_series.str.strip(' ,')
loan_series = loan_series.mask(loan_series.eq(''))
loan_series = pd.Series(
    np.where(loan_series.isna() & df['Num_of_Loan'].fillna(0).le(0), 'Sin_prestamo', loan_series),
    index=df.index,
    dtype='string',
)
loan_series = pd.Series(
    np.where(loan_series.isna() & df['Num_of_Loan'].fillna(0).gt(0), 'Prestamo_no_especificado', loan_series),
    index=df.index,
    dtype='string',
)
df[loan_col] = loan_series

df[target_col] = df[target_col].astype('Int64')

text_view_after = df.astype('string')
quality_after = pd.DataFrame(
    {
        'column': df.columns,
        'dtype': df.dtypes.astype(str).to_numpy(),
        'missing_count': df.isna().sum().to_numpy(),
        'missing_pct': (df.isna().mean() * 100).to_numpy(),
        'blank_like_count': text_view_after.apply(lambda s: s.str.strip().eq('').fillna(False).sum()).to_numpy(),
        'NM_count': text_view_after.apply(lambda s: s.eq('NM').fillna(False).sum()).to_numpy(),
        'Not_Specified_count': text_view_after.apply(lambda s: s.eq('Not Specified').fillna(False).sum()).to_numpy(),
        'unique_values': df.nunique(dropna=True).to_numpy(),
    }
).sort_values(['missing_count', 'blank_like_count'], ascending=False).reset_index(drop=True)

numeric_view_after = df[numeric_columns + [target_col]].apply(pd.to_numeric, errors='coerce')
suspicious_after = pd.DataFrame(
    {
        'check': [
            'Age < 18',
            'Delay_from_due_date < 0',
            'Num_Bank_Accounts con decimales',
            'Num_Credit_Card con decimales',
            'Num_of_Loan con decimales',
            'Num_of_Delayed_Payment con decimales',
            'Changed_Credit_Limit con decimales',
            'Num_Credit_Inquiries con decimales',
        ],
        'count': [
            int(numeric_view_after['Age'].lt(18).fillna(False).sum()),
            int(numeric_view_after['Delay_from_due_date'].lt(0).fillna(False).sum()),
            int(((numeric_view_after['Num_Bank_Accounts'].dropna() % 1) != 0).sum()),
            int(((numeric_view_after['Num_Credit_Card'].dropna() % 1) != 0).sum()),
            int(((numeric_view_after['Num_of_Loan'].dropna() % 1) != 0).sum()),
            int(((numeric_view_after['Num_of_Delayed_Payment'].dropna() % 1) != 0).sum()),
            int(((numeric_view_after['Changed_Credit_Limit'].dropna() % 1) != 0).sum()),
            int(((numeric_view_after['Num_Credit_Inquiries'].dropna() % 1) != 0).sum()),
        ],
    }
).sort_values('count', ascending=False).reset_index(drop=True)

comparison_summary = pd.DataFrame(
    {
        'metric': [
            'Filas',
            'Columnas',
            'Nulos totales',
            'Celdas con blanco',
            'Celdas con "NM"',
            'Celdas con "Not Specified"',
        ],
        'before': [
            int(raw_df.shape[0]),
            int(raw_df.shape[1]),
            int(raw_df.isna().sum().sum()),
            int(text_view_before.apply(lambda s: s.str.strip().eq('').fillna(False).sum()).sum()),
            int(text_view_before.apply(lambda s: s.eq('NM').fillna(False).sum()).sum()),
            int(text_view_before.apply(lambda s: s.eq('Not Specified').fillna(False).sum()).sum()),
        ],
        'after': [
            int(df.shape[0]),
            int(df.shape[1]),
            int(df.isna().sum().sum()),
            int(text_view_after.apply(lambda s: s.str.strip().eq('').fillna(False).sum()).sum()),
            int(text_view_after.apply(lambda s: s.eq('NM').fillna(False).sum()).sum()),
            int(text_view_after.apply(lambda s: s.eq('Not Specified').fillna(False).sum()).sum()),
        ],
    }
)


## 4. Primera ejecución end-to-end

In [ ]:
# Mostramos el efecto de la limpieza y dejamos el dataframe final listo para EDA/PCA/ANN.
print('Resumen antes vs despues de la limpieza:')
display(comparison_summary)

print('Variables con mayor volumen de calidad problematica antes de limpiar:')
display(quality_before.head(10))

print('Variables con mayor volumen de calidad problematica despues de limpiar:')
display(quality_after.head(10))

print('Chequeos de valores sospechosos antes de limpiar:')
display(suspicious_before)

print('Chequeos de valores sospechosos despues de limpiar:')
display(suspicious_after)

print('Vista previa del dataset limpio:')
display(df.head())


## 5. Inspección de resultados intermedios

In [ ]:
# EDA principal para justificar decisiones de preprocesamiento.
target_distribution = (
    df[target_col]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis('Credit_Score')
    .reset_index(name='frecuencia')
)
target_distribution['proporcion'] = target_distribution['frecuencia'] / target_distribution['frecuencia'].sum()

missing_before = quality_before[['column', 'missing_count', 'blank_like_count', 'NM_count', 'Not_Specified_count']].copy()
missing_before['total_problemas_texto'] = (
    missing_before['missing_count']
    + missing_before['blank_like_count']
    + missing_before['NM_count']
    + missing_before['Not_Specified_count']
)
missing_before = missing_before.sort_values('total_problemas_texto', ascending=False).head(10)

numeric_profile = df[numeric_columns].describe().T
correlation_matrix = df[numeric_columns + [target_col]].corr(numeric_only=True)

loan_indicator = df[loan_col].str.get_dummies(sep=',')
loan_indicator.columns = loan_indicator.columns.str.strip()
loan_indicator = loan_indicator.T.groupby(level=0).max().T
loan_frequency = (
    loan_indicator
    .sum()
    .sort_values(ascending=False)
    .rename_axis('loan_type')
    .reset_index(name='frecuencia')
)

categorical_tables = {
    column: df[column].value_counts(dropna=False).head(10)
    for column in base_categorical_columns
}

print('Distribucion del target:')
display(target_distribution)

print('Perfil estadistico de variables numericas:')
display(numeric_profile)

print('Frecuencias principales de Type_of_Loan ya normalizado:')
display(loan_frequency.head(15))

for column, table in categorical_tables.items():
    print(f'Frecuencias principales de {column}:')
    display(table.to_frame(name='frecuencia'))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(target_distribution['Credit_Score'].astype(str), target_distribution['frecuencia'], color=['#4C72B0', '#55A868', '#C44E52'])
axes[0, 0].set_title('Distribucion de Credit_Score')
axes[0, 0].set_xlabel('Clase')
axes[0, 0].set_ylabel('Frecuencia')

axes[0, 1].barh(missing_before['column'], missing_before['total_problemas_texto'], color='#8172B2')
axes[0, 1].set_title('Top 10 variables con problemas de calidad')
axes[0, 1].set_xlabel('Cantidad de problemas detectados')

selected_numeric = ['Age', 'Annual_Income', 'Outstanding_Debt', 'Monthly_Balance']
for idx, column in enumerate(selected_numeric[:2]):
    df[column].plot(kind='hist', bins=30, ax=axes[1, idx], alpha=0.8)
    axes[1, idx].set_title(f'Distribucion de {column}')
    axes[1, idx].set_xlabel(column)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for class_value in sorted(df[target_col].dropna().unique()):
    subset = df.loc[df[target_col] == class_value, 'Outstanding_Debt']
    axes[0].hist(subset, bins=25, alpha=0.45, label=f'Clase {class_value}')
axes[0].set_title('Outstanding_Debt por clase')
axes[0].set_xlabel('Outstanding_Debt')
axes[0].legend()

for class_value in sorted(df[target_col].dropna().unique()):
    subset = df.loc[df[target_col] == class_value, 'Credit_Utilization_Ratio']
    axes[1].hist(subset, bins=25, alpha=0.45, label=f'Clase {class_value}')
axes[1].set_title('Credit_Utilization_Ratio por clase')
axes[1].set_xlabel('Credit_Utilization_Ratio')
axes[1].legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 9))
heatmap = ax.imshow(correlation_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=90)
ax.set_yticks(range(len(correlation_matrix.index)))
ax.set_yticklabels(correlation_matrix.index)
ax.set_title('Matriz de correlacion numerica')
plt.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 6. PCA sobre la matriz completa preprocesada

In [ ]:
# Construimos la matriz de modelado despues de limpiar, estandarizamos y aplicamos PCA.
loan_dummies = df[loan_col].str.get_dummies(sep=',')
loan_dummies.columns = loan_dummies.columns.str.strip()
loan_dummies = loan_dummies.T.groupby(level=0).max().T
loan_dummies = loan_dummies.rename(columns=lambda value: f"Loan__{value.replace(' ', '_')}").astype(float)

categorical_dummies = pd.get_dummies(df[base_categorical_columns], prefix=base_categorical_columns, dtype=float)
numeric_block = df[numeric_columns].astype(float)

X_model = pd.concat([numeric_block, categorical_dummies, loan_dummies], axis=1)
y = df[target_col].astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_model)

pca_model = PCA()
X_pca = pca_model.fit_transform(X_scaled)
pca_n_components = 35
X_pca_35 = X_pca[:, :pca_n_components]

explained_variance = pd.DataFrame(
    {
        'Componente': np.arange(1, len(pca_model.explained_variance_ratio_) + 1),
        'Varianza_Explicada': pca_model.explained_variance_ratio_,
        'Varianza_Acumulada': np.cumsum(pca_model.explained_variance_ratio_),
    }
)

variance_targets = {
    'Componentes_80pct': int(np.argmax(explained_variance['Varianza_Acumulada'].to_numpy() >= 0.80) + 1),
    'Componentes_90pct': int(np.argmax(explained_variance['Varianza_Acumulada'].to_numpy() >= 0.90) + 1),
    'Componentes_95pct': int(np.argmax(explained_variance['Varianza_Acumulada'].to_numpy() >= 0.95) + 1),
}
selected_variance_35 = float(explained_variance.loc[explained_variance['Componente'] == pca_n_components, 'Varianza_Acumulada'].iloc[0])

projection_df = pd.DataFrame(
    {
        'PC1': X_pca[:, 0],
        'PC2': X_pca[:, 1],
        'Credit_Score': y.astype(str).to_numpy(),
    }
)

loadings = pd.DataFrame(
    pca_model.components_[:5].T,
    index=X_model.columns,
    columns=[f'PC{i}' for i in range(1, 6)],
)

top_features_pc1 = loadings['PC1'].abs().sort_values(ascending=False).head(15).index
top_features_pc2 = loadings['PC2'].abs().sort_values(ascending=False).head(15).index

print('Dimension final de la matriz para modelado:', X_model.shape)
print(f'Cantidad de componentes seleccionados para la ANN: {pca_n_components}')
print(f'Varianza acumulada con {pca_n_components} componentes: {selected_variance_35:.2%}')
print('Numero de componentes necesarios por umbral de varianza:')
display(pd.DataFrame(list(variance_targets.items()), columns=['criterio', 'componentes']))

print('Primeras filas de la varianza explicada:')
display(explained_variance.head(10))

print('Variables con mayor peso absoluto en PC1:')
display(loadings.loc[top_features_pc1, ['PC1']].sort_values('PC1', key=lambda s: s.abs(), ascending=False))

print('Variables con mayor peso absoluto en PC2:')
display(loadings.loc[top_features_pc2, ['PC2']].sort_values('PC2', key=lambda s: s.abs(), ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(explained_variance['Componente'], explained_variance['Varianza_Explicada'], marker='o')
axes[0].set_title('Scree plot')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada')

axes[1].plot(explained_variance['Componente'], explained_variance['Varianza_Acumulada'], marker='o', color='#55A868')
axes[1].axhline(0.80, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(0.90, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(0.95, color='gray', linestyle='--', linewidth=1)
axes[1].axvline(pca_n_components, color='#4C72B0', linestyle='--', linewidth=1)
axes[1].set_title('Varianza acumulada')
axes[1].set_xlabel('Componente principal')
axes[1].set_ylabel('Varianza acumulada')

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 7))
colors = {'0': '#4C72B0', '1': '#55A868', '2': '#C44E52'}
for class_value, group in projection_df.groupby('Credit_Score'):
    ax.scatter(group['PC1'], group['PC2'], s=18, alpha=0.55, label=f'Clase {class_value}', color=colors.get(class_value, '#8172B2'))
ax.set_title('Proyeccion 2D del PCA por Credit_Score')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
plt.tight_layout()
plt.show()


## 7. ANN multiclass sobre PCA


In [ ]:
# Entrenamos una ANN multiclase con salida softmax sobre las 35 componentes principales.
X_train, X_temp, y_train, y_temp = train_test_split(
    X_pca_35,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

class_weights_array = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight = {int(label): float(weight) for label, weight in zip(np.unique(y_train), class_weights_array)}

ann_model = keras.Sequential(
    [
        layers.Input(shape=(pca_n_components,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.25),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.15),
        layers.Dense(16, activation='relu'),
        layers.Dense(3, activation='softmax'),
    ],
    name='ann_credit_score_multiclass',
)
ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5),
]

history = ann_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=0,
)

history_df = pd.DataFrame(history.history)
history_df.index = history_df.index + 1
history_df.index.name = 'epoch'

y_pred_proba = ann_model.predict(X_test, verbose=0)
y_pred = y_pred_proba.argmax(axis=1)
test_loss, test_accuracy = ann_model.evaluate(X_test, y_test, verbose=0)

report_df = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T
conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=[f'Real_{label}' for label in sorted(np.unique(y))],
    columns=[f'Pred_{label}' for label in sorted(np.unique(y))],
)

split_summary = pd.DataFrame(
    {
        'split': ['train', 'validation', 'test'],
        'filas': [len(y_train), len(y_val), len(y_test)],
        'proporcion': [len(y_train) / len(y), len(y_val) / len(y), len(y_test) / len(y)],
    }
)

summary_lines = []
ann_model.summary(print_fn=summary_lines.append)
print('Arquitectura de la ANN:')
print('\n'.join(summary_lines))
print('\nResumen de particiones:')
display(split_summary)
print('Pesos de clase usados en entrenamiento:')
display(pd.Series(class_weight, name='peso').rename_axis('clase').to_frame())
print(f'Loss en test: {test_loss:.4f}')
print(f'Accuracy en test: {test_accuracy:.4f}')
print('Metricas de clasificacion por clase:')
display(report_df.loc[['0', '1', '2'], ['precision', 'recall', 'f1-score', 'support']])
print('Matriz de confusion:')
display(conf_matrix)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_df.index, history_df['loss'], label='train_loss')
axes[0].plot(history_df.index, history_df['val_loss'], label='val_loss')
axes[0].set_title('Curva de perdida')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df.index, history_df['accuracy'], label='train_accuracy')
axes[1].plot(history_df.index, history_df['val_accuracy'], label='val_accuracy')
axes[1].set_title('Curva de accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
heatmap = ax.imshow(conf_matrix, cmap='Blues')
ax.set_xticks(range(conf_matrix.shape[1]))
ax.set_xticklabels(conf_matrix.columns, rotation=45, ha='right')
ax.set_yticks(range(conf_matrix.shape[0]))
ax.set_yticklabels(conf_matrix.index)
ax.set_title('Matriz de confusion ANN')
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(j, i, int(conf_matrix.iloc[i, j]), ha='center', va='center', color='black')
plt.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 8. Validaciones básicas y guardado de artefactos


In [ ]:
# Validamos supuestos minimos y guardamos el CSV limpio y los artefactos de la ANN.
assert target_col in raw_df.columns, 'La variable objetivo no existe en el dataset.'
assert set(y.unique()) == {0, 1, 2}, 'El target no contiene exactamente las tres clases esperadas.'
assert X_model.isna().sum().sum() == 0, 'La matriz final para PCA/ANN contiene valores faltantes.'
assert X_scaled.shape == X_model.shape, 'La matriz escalada no conserva la forma esperada.'
assert X_pca_35.shape == (df.shape[0], pca_n_components), 'La matriz PCA para ANN no tiene la forma esperada.'
assert projection_df.shape[0] == df.shape[0], 'La proyeccion PCA no coincide con el numero de observaciones.'
assert ann_model.output_shape[-1] == 3, 'La capa de salida de la ANN no tiene 3 neuronas.'
assert int(text_view_after.apply(lambda s: s.eq('NM').fillna(False).sum()).sum()) == 0, 'Aun quedan tokens NM en el dataframe limpio.'
assert int(text_view_after.apply(lambda s: s.eq('Not Specified').fillna(False).sum()).sum()) == 0, 'Aun quedan tokens Not Specified en el dataframe limpio.'
assert np.allclose(y_pred_proba.sum(axis=1), 1.0, atol=1e-5), 'Softmax no esta generando probabilidades validas.'
assert report_df.loc[['0', '1', '2'], 'support'].sum() == len(y_test), 'El reporte de clasificacion no cubre todo el set de test.'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(ARTIFACTS_DIR / 'riesgo_limpio.csv', index=False)
ann_model.save(ARTIFACTS_DIR / 'ann_multiclass.keras')
history_df.to_csv(ARTIFACTS_DIR / 'ann_history.csv')
report_df.to_csv(ARTIFACTS_DIR / 'ann_test_report.csv')

print(f'CSV limpio guardado en: {ARTIFACTS_DIR / "riesgo_limpio.csv"}')
print(f'Modelo ANN guardado en: {ARTIFACTS_DIR / "ann_multiclass.keras"}')
print(f'Historia de entrenamiento guardada en: {ARTIFACTS_DIR / "ann_history.csv"}')
print(f'Reporte de test guardado en: {ARTIFACTS_DIR / "ann_test_report.csv"}')


## 9. Análisis de resultados y decisiones


In [ ]:
# Resumen interpretativo del pipeline EDA + PCA + ANN.
class_metrics = report_df.loc[['0', '1', '2'], ['precision', 'recall', 'f1-score', 'support']].copy()
best_f1_class = class_metrics['f1-score'].astype(float).idxmax()
lowest_recall_class = class_metrics['recall'].astype(float).idxmin()
majority_class = int(target_distribution.loc[target_distribution['frecuencia'].idxmax(), 'Credit_Score'])
minority_class = int(target_distribution.loc[target_distribution['frecuencia'].idxmin(), 'Credit_Score'])

analysis_summary = pd.DataFrame(
    {
        'hallazgo': [
            'Balance del target',
            'Limpieza aplicada',
            'PCA seleccionado',
            'Arquitectura ANN',
            'Mejor clase por F1',
            'Clase mas retadora por recall',
            'Desempeno general',
        ],
        'detalle': [
            f'La clase {majority_class} es la mas frecuente y la clase {minority_class} la menos frecuente.',
            'La limpieza elimino blancos, NM y Not Specified usando operaciones vectorizadas de pandas/numpy.',
            f'Se usaron {pca_n_components} componentes principales que conservan {selected_variance_35:.2%} de la varianza acumulada.',
            'La red usa capas densas 64-32-16 con ReLU, regularizacion con Dropout y salida Dense(3, softmax).',
            f'La clase {best_f1_class} obtuvo el F1-score mas alto en test.',
            f'La clase {lowest_recall_class} obtuvo el recall mas bajo y merece ajustes adicionales si se quiere mejorar sensibilidad.',
            f'La ANN sobre PCA alcanzo accuracy de test = {test_accuracy:.4f} y valida el uso de un modelo no lineal para las 3 clases.',
        ],
    }
)

print('Resumen final del taller:')
display(analysis_summary)

print('Metricas detalladas por clase:')
display(class_metrics)
